# 02_rank_accuracy_tradeoff（corrected）

原本 `02_rank_accuracy_tradeoff.ipynb` は **test set で rank を選んでいた** 歴史的実験として残す。
この corrected 版では役割を分ける。

```text
train（学習）
  ↓
validation（rank 選択・トレードオフの可視化）
  ↓
選んだ 1 組だけ test（最終確認）
```

test を rank 選択に使わない。保存先も原本と別にする。


## 1. import

定型処理は `src/nn_compression/`。この Notebook には実験条件と解釈を残す。


In [ ]:
from __future__ import annotations

import itertools
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

for _candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _src = _candidate / "src"
    if (_src / "nn_compression").is_dir():
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from nn_compression.compression import make_two_layer_svd_model, retained_energy
from nn_compression.datasets import shuffled_index_splits
from nn_compression.metrics import (
    accuracy_drop,
    agreement,
    benchmark_inference,
    count_parameters,
    estimate_mlp_macs,
    logits_rmse,
    parameters_reduction,
    take_inference_batch,
)
from nn_compression.models import MNISTMLP
from nn_compression.training import (
    evaluate,
    fit_with_early_stopping,
    non_shuffling_loader,
)
from nn_compression.utils import find_project_root, get_experiment_dirs, set_seed


## 2. 実験条件

学習は短い（原本と同じ 5 epoch）。rank 候補も原本に合わせる。


In [ ]:
SEED = 0
LEARNING_RATE = 1e-3
MAX_EPOCHS = 5
PATIENCE = 5
MIN_DELTA = 1e-4
BATCH_SIZE = 64
R1_LIST = [16, 32, 64, 128]
R2_LIST = [16, 32, 64, 128]
BENCH_WARMUP = 20
BENCH_REPEATS = 200

DATASET_NAME = "10_mnist_mlp"
EXPERIMENT_NAME = "02_rank_accuracy_tradeoff_corrected"

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
project_root = find_project_root(Path.cwd())
data_dir, models_dir, results_dir = get_experiment_dirs(
    project_root,
    DATASET_NAME,
    EXPERIMENT_NAME,
)
print("device:", device)
print("results_dir:", results_dir)


## 3. データ分割

公式 train 60,000 を 50,000 / 5,000 / 5,000 に分ける。
最後の 5,000 が **rank 選択用 validation**。test 10,000 は最後まで触らない。


In [ ]:
transform = transforms.ToTensor()
full_train = datasets.MNIST(
    root=data_dir,
    train=True,
    download=True,
    transform=transform,
)
test_dataset = datasets.MNIST(
    root=data_dir,
    train=False,
    download=True,
    transform=transform,
)

train_idx, val_fit_idx, val_rank_idx = shuffled_index_splits(
    len(full_train),
    (50_000, 5_000, 5_000),
    seed=SEED,
)
train_dataset = Subset(full_train, train_idx)
val_fit_dataset = Subset(full_train, val_fit_idx)
val_rank_dataset = Subset(full_train, val_rank_idx)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)
train_eval_loader = non_shuffling_loader(train_loader)
val_fit_loader = DataLoader(val_fit_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_rank_loader = DataLoader(val_rank_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print("train", len(train_dataset), "val_fit", len(val_fit_dataset))
print("val_rank", len(val_rank_dataset), "test", len(test_dataset))


## 4. Baseline 学習

Early Stopping の validation は rank 選択用とは別セット。


In [ ]:
set_seed(SEED)
loader_generator.manual_seed(SEED)
model = MNISTMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
fit = fit_with_early_stopping(
    model,
    train_loader,
    val_fit_loader,
    criterion,
    optimizer,
    device,
    MAX_EPOCHS,
    PATIENCE,
    MIN_DELTA,
    train_eval_loader=train_eval_loader,
    log_every_epoch=True,
)
model = fit["model"]
print("best epoch", fit["best_epoch"], "val loss", fit["best_validation_loss"])


## 5. Rank sweep（validation のみ）

各 `(fc1_rank, fc2_rank)` の精度は **val_rank** で見る。
推論時間は同じ `input_batch`・warmup・repeats で測る。
圧縮モデルは表に残さず、必要なときだけ再構築する。


In [ ]:
rank_configs = list(itertools.product(R1_LIST, R2_LIST))
inference_batch = take_inference_batch(val_rank_loader)
baseline_bench = benchmark_inference(
    model,
    device=device,
    warmup=BENCH_WARMUP,
    repeats=BENCH_REPEATS,
    input_batch=inference_batch,
    return_details=True,
)
baseline_time_s = baseline_bench["time_s"]
baseline_loss, baseline_acc = evaluate(model, val_rank_loader, criterion, device)
print(
    "val acc",
    f"{baseline_acc:.4f}",
    "batch",
    baseline_bench["batch_size"],
    "shape",
    baseline_bench["input_shape"],
)

rank_results = []
for fc1_rank, fc2_rank in rank_configs:
    compressed = make_two_layer_svd_model(
        model,
        fc1_rank=fc1_rank,
        fc2_rank=fc2_rank,
    )
    val_loss, val_acc = evaluate(compressed, val_rank_loader, criterion, device)
    _, compressed_macs, compute_reduction = estimate_mlp_macs(
        model, fc1_rank, fc2_rank, verbose=False
    )
    compressed_time_s = benchmark_inference(
        compressed,
        device=device,
        warmup=BENCH_WARMUP,
        repeats=BENCH_REPEATS,
        input_batch=inference_batch,
    )
    rank_results.append(
        {
            "fc1_rank": fc1_rank,
            "fc2_rank": fc2_rank,
            "parameters": count_parameters(compressed),
            "parameters_reduction": parameters_reduction(model, compressed),
            "validation_loss": val_loss,
            "validation_acc": val_acc,
            "accuracy_drop": accuracy_drop(baseline_acc, val_acc, verbose=False),
            "compressed_macs": compressed_macs,
            "compute_reduction": compute_reduction,
            "baseline_time_ms": baseline_time_s * 1000,
            "compressed_time_ms": compressed_time_s * 1000,
            "agreement": agreement(model, compressed, val_rank_loader, device),
            "logits_rmse": logits_rmse(model, compressed, val_rank_loader, device),
            "retained_energy_fc1": retained_energy(model.fc1, fc1_rank),
            "retained_energy_fc2": retained_energy(model.fc2, fc2_rank),
        }
    )
    print(
        f"r1={fc1_rank:3d} r2={fc2_rank:3d} | "
        f"val_acc={val_acc:.4f} params={count_parameters(compressed)}"
    )

df = pd.DataFrame(rank_results).sort_values(
    ["validation_loss", "parameters"]
).reset_index(drop=True)
print(df.head())
results_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(results_dir / "rank_sweep_validation.csv", index=False)


## 6. 可視化（validation）

ヒートマップの値は validation accuracy。test はまだ使わない。


In [ ]:
acc_pivot = df.pivot(
    index="fc1_rank",
    columns="fc2_rank",
    values="validation_acc",
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(acc_pivot.values, origin="lower", aspect="auto")
axes[0].set_xticks(range(len(acc_pivot.columns)), acc_pivot.columns)
axes[0].set_yticks(range(len(acc_pivot.index)), acc_pivot.index)
axes[0].set_xlabel("fc2_rank")
axes[0].set_ylabel("fc1_rank")
axes[0].set_title("validation accuracy")
fig.colorbar(im, ax=axes[0])
axes[1].scatter(df["parameters"], df["validation_loss"])
axes[1].set_xlabel("parameters")
axes[1].set_ylabel("validation_loss")
axes[1].set_title("val loss vs params")
plt.tight_layout()
fig.savefig(results_dir / "rank_accuracy_tradeoff_validation.png", dpi=150)
plt.show()


## 7. 最終確認だけ test

validation_loss が最小の 1 組を選び、ここで初めて test を見る。


In [ ]:
best = df.iloc[0]
fc1_rank = int(best["fc1_rank"])
fc2_rank = int(best["fc2_rank"])
print("selected from validation:", fc1_rank, fc2_rank)

final_model = make_two_layer_svd_model(
    model,
    fc1_rank=fc1_rank,
    fc2_rank=fc2_rank,
)
baseline_test_loss, baseline_test_acc = evaluate(
    model, test_loader, criterion, device
)
final_test_loss, final_test_acc = evaluate(
    final_model, test_loader, criterion, device
)
summary = pd.DataFrame(
    [
        {
            "model": "baseline",
            "fc1_rank": "-",
            "fc2_rank": "-",
            "test_loss": baseline_test_loss,
            "test_acc": baseline_test_acc,
        },
        {
            "model": "selected SVD",
            "fc1_rank": fc1_rank,
            "fc2_rank": fc2_rank,
            "test_loss": final_test_loss,
            "test_acc": final_test_acc,
        },
    ]
)
print(summary)
summary.to_csv(results_dir / "test_comparison.csv", index=False)
torch.save(final_model.state_dict(), models_dir / "selected_from_validation.pt")
